# CSMorgan MEDVQA-GI 2026 — Task 2: Explainability + Safety

All outputs saved to **Google Drive** `/MyDrive/CSMORGAN_MEDVQA_2026/`

**Submission**: Email ZIP to `steven@simula.no` — NOT medvqa CLI

**Subject**: `ImageCLEFmed-MEDVQA-GI-2026 Task2 Submission - CSMorgan-MEDVQA`

**Run order**: 01 → restart → 02 → 03 → 04 → 05 → 06 → 07 → 08 → 09 → 10


In [ ]:
\
# ── CELL 01: Install Dependencies ────────────────────────────────────────────
# HPC-safe: falls back to --user if system Python is read-only.
import subprocess, sys, site

PACKAGES = [
    "ms-swift==3.8.0", "bitsandbytes", "qwen_vl_utils==0.0.11",
    "datasets", "transformers>=4.40.0", "peft", "accelerate",
    "evaluate", "nltk", "rouge_score", "tqdm",
    "Pillow", "pandas", "huggingface_hub", "matplotlib",
]

def pip_live(packages, extra=None):
    cmd = [sys.executable, "-m", "pip", "install"] + (extra or []) + packages
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    out = []
    for line in proc.stdout:
        out.append(line)
        s = line.rstrip()
        if s and "DEPRECATION" not in s and s.startswith(("Collecting","Installing","Successfully","ERROR","error")):
            print(s)
    proc.wait()
    return proc.returncode, "".join(out)

rc, out = pip_live(PACKAGES)
if rc != 0 and ("Permission denied" in out or "Errno 13" in out):
    print("Retrying with --user ...")
    rc, out = pip_live(PACKAGES, extra=["--user"])
    if rc == 0:
        u = site.getusersitepackages()
        if u not in sys.path: sys.path.insert(0, u)
        print(f"Installed to {u}")
    else:
        raise RuntimeError("pip install failed")
print("Install complete. RESTART KERNEL NOW if first run.")


Successfully built rouge_score oss2 transformers-stream-generator aliyun-python-sdk-core crcmod
Install complete. RESTART KERNEL NOW if first run.


In [ ]:
# ── CELL 02: Google Drive Mount & Directory Structure ─────────────────────────
# All outputs (images, checkpoints, results, logs, plots) saved to Google Drive.
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    ROOT = Path("/content/drive/MyDrive/CSMORGAN_MEDVQA_2026")
    print("Google Drive mounted.")
except ImportError:
    ROOT = Path("./CSMORGAN_MEDVQA_2026")
    print("Local mode — saving to ./CSMORGAN_MEDVQA_2026")

IMG_DIR     = ROOT / "data"  / "images"
JSONL_DIR   = ROOT / "data"  / "jsonl"
CKPT_DIR    = ROOT / "checkpoints"
RESULTS_DIR = ROOT / "results"
LOGS_DIR    = ROOT / "logs"
PLOTS_DIR   = ROOT / "plots"
HF_DIR      = ROOT / "hf_export"

for d in [IMG_DIR, JSONL_DIR, CKPT_DIR, RESULTS_DIR, LOGS_DIR, PLOTS_DIR, HF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Root: {ROOT}")
print("Directories:")
for name, path in [("images",IMG_DIR),("jsonl",JSONL_DIR),("checkpoints",CKPT_DIR),
                    ("results",RESULTS_DIR),("logs",LOGS_DIR),("plots",PLOTS_DIR)]:
    print(f"  {name:14s}: {path}")


Mounted at /content/drive
Google Drive mounted.
Root: /content/drive/MyDrive/CSMORGAN_MEDVQA_2026
Directories:
  images        : /content/drive/MyDrive/CSMORGAN_MEDVQA_2026/data/images
  jsonl         : /content/drive/MyDrive/CSMORGAN_MEDVQA_2026/data/jsonl
  checkpoints   : /content/drive/MyDrive/CSMORGAN_MEDVQA_2026/checkpoints
  results       : /content/drive/MyDrive/CSMORGAN_MEDVQA_2026/results
  logs          : /content/drive/MyDrive/CSMORGAN_MEDVQA_2026/logs
  plots         : /content/drive/MyDrive/CSMORGAN_MEDVQA_2026/plots


In [ ]:
\
# ── CELL 03: Imports & Config ─────────────────────────────────────────────────
import gc, json, os, random, re, site, subprocess, sys, tempfile, time
from pathlib import Path
from typing import Dict, List, Optional

import nltk
import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

for p in ("punkt","punkt_tab","wordnet","omw-1.4"):
    nltk.download(p, quiet=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
VRAM_GB = 0.0
if torch.cuda.is_available():
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {VRAM_GB:.1f} GB")

# HPC: ensure user site-packages visible to this kernel
_us = site.getusersitepackages()
if _us not in sys.path: sys.path.insert(0, _us)

# Model registry
MODELS = {
    "qwen": {
        "base":    "Qwen/Qwen2.5-VL-7B-Instruct",
        "adapter": "SimulaMet/Qwen2.5-VL-KvasirVQA-x1-ft",
        "label":   "Qwen2.5-VL-7B-KvasirFT",
    },
    "qwen_transf": {
        "base":    "Qwen/Qwen2.5-VL-7B-Instruct",
        "adapter": "SimulaMet/Qwen2.5-VL-Transf-KvasirVQA-x1-ft",
        "label":   "Qwen2.5-VL-7B-TransfFT",
    },
}

MAX_EVAL = None   # None = full test set | set 300 for quick run
print(f"Config ready. MAX_EVAL={MAX_EVAL}")


GPU : NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB
Config ready. MAX_EVAL=None


In [ ]:
\
# ── CELL 04: Data Loading ─────────────────────────────────────────────────────
from datasets import load_dataset

print("Downloading images from SimulaMet-HOST/Kvasir-VQA ...")
ds_host  = load_dataset("SimulaMet-HOST/Kvasir-VQA", split="raw")
_, u_idx = np.unique(ds_host["img_id"], return_index=True)
ds_uniq  = ds_host.select(sorted(u_idx))
existing = {p.stem for p in IMG_DIR.glob("*.jpg")}
saved = 0
for row in tqdm(ds_uniq, desc="Saving images", leave=False):
    if row["img_id"] in existing: continue
    row["image"].save(IMG_DIR / f"{row['img_id']}.jpg"); saved += 1
print(f"  {saved} new images | {len(list(IMG_DIR.glob('*.jpg')))} total")

print("\\nLoading QA pairs from SimulaMet/Kvasir-VQA-x1 ...")
ds_x1 = load_dataset("SimulaMet/Kvasir-VQA-x1")
for spl in ds_x1:
    print(f"  {spl}: {len(ds_x1[spl]):,} rows")

test_samples = list(ds_x1["test"])
if MAX_EVAL: test_samples = test_samples[:MAX_EVAL]

import pandas as pd
meta_df = pd.concat([
    ds_x1[spl].to_pandas().drop(columns=["image"], errors="ignore").assign(split=spl)
    for spl in ds_x1], ignore_index=True)
train_df = meta_df[meta_df["split"]=="train"].reset_index(drop=True)

missing = {r["img_id"] for r in test_samples} - {p.stem for p in IMG_DIR.glob("*.jpg")}
print(f"\\nTest samples : {len(test_samples):,}")
print(f"Images present: {len(list(IMG_DIR.glob('*.jpg'))):,}")
print(f"Missing       : {len(missing)} {'OK' if not missing else 'WARNING'}")


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

data/00000.parquet:   0%|          | 0.00/26.8M [00:00<?, ?B/s]

data/00001.parquet:   0%|          | 0.00/26.2M [00:00<?, ?B/s]

...

data/train-00000-of-00001.parquet:   0%|          | 0.00/14.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/143594 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15955 [00:00<?, ? examples/s]

  train: 143,594 rows
  test: 15,955 rows
\nTest samples : 15,955
Images present: 6,500
Missing       : 0 OK


In [ ]:
from huggingface_hub import login
login()

In [ ]:
# ── CELL 08: Fine-tune MedGemma-4B (QLoRA, S2 safety format) ─────────────────
import os, site, subprocess, sys, time, json, random
from pathlib import Path
from tqdm.auto import tqdm

DRY_RUN   = False
MAX_STEPS = 800
MAX_LEN   = 2048

random.seed(42)

assert DRY_RUN in (True, False)
print(f"DRY_RUN  = {DRY_RUN}")
print(f"MAX_STEPS= {MAX_STEPS}")

# ── JSONL helpers ─────────────────────────────────────────────────────────────
def _is_valid_json(line: str) -> bool:
    try:
        json.loads(line)
        return True
    except Exception:
        return False

def _validate_and_fix(path):
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()
    bad = [i for i, l in enumerate(lines) if not _is_valid_json(l)]
    if bad:
        print(f"  ⚠️  {len(bad)} bad lines — fixing ...")
        valid = [l for l in lines if _is_valid_json(l)]
        with open(path, "w", encoding="utf-8", newline="\n") as f:
            f.writelines(valid)
        print(f"  Fixed: {len(valid):,} lines kept")
    else:
        print(f"  ✅  All {len(lines):,} lines valid JSON")

# ── Build S2 JSONL (stratified subset — same logic as Cell 05) ────────────────
S2_SYS = (
    "You are a clinically trained GI endoscopy AI. "
    "For every question provide exactly:\n"
    "ANSWER: <concise answer>\n"
    "JUSTIFICATION: <evidence-based clinical reasoning referencing visible findings>\n"
    "CONFIDENCE: <float 0.0-0.85, never exceed 0.85>\n"
    "SAFETY_NOTE: <clinical concern or None>"
)

def s2_target(answer: str) -> str:
    return (
        f"ANSWER: {answer}\n"
        "JUSTIFICATION: Based on the visible endoscopic findings in the image.\n"
        "CONFIDENCE: 0.80\n"
        "SAFETY_NOTE: None"
    )

def build_s2_subset(n_per_complexity=5000):
    """15K stratified S2 subset — enough for 800 steps ~1.6x coverage."""
    out = JSONL_DIR / "s2_subset.jsonl"
    if out.exists():
        lines = open(out, encoding="utf-8").readlines()
        bad   = sum(1 for l in lines if not _is_valid_json(l))
        if bad == 0 and len(lines) > 100:
            print(f"  s2_subset.jsonl exists and valid: {len(lines):,} rows — skipping")
            return out
        print(f"  s2_subset.jsonl corrupted — rebuilding")
        out.unlink()

    print("  Loading rows for S2 subset ...")
    existing = {p.stem for p in IMG_DIR.glob("*.jpg")}
    by_level = {1: [], 2: [], 3: []}
    for spl in ["train", "test"]:
        for row in ds_x1[spl]:
            if row["img_id"] not in existing:
                continue
            lvl = int(row.get("complexity") or 1)
            by_level[lvl].append(row)

    print(f"  Available: L1={len(by_level[1]):,}  "
          f"L2={len(by_level[2]):,}  L3={len(by_level[3]):,}")

    sampled = []
    for lvl in [1, 2, 3]:
        rows = by_level[lvl]
        n    = min(n_per_complexity, len(rows))
        sampled.extend(random.sample(rows, n))
        print(f"  Sampled L{lvl}: {n:,}")

    sampled.sort(key=lambda r: int(r.get("complexity") or 1))
    print(f"  Total: {len(sampled):,} rows")

    count = 0
    with open(out, "w", encoding="utf-8", newline="\n") as f:
        for row in tqdm(sampled, desc="Writing s2_subset", leave=False):
            ip = IMG_DIR / f"{row['img_id']}.jpg"
            f.write(json.dumps({
                "messages": [
                    {"role": "system",    "content": S2_SYS},
                    {"role": "user",      "content": f"<image>{row['question']}"},
                    {"role": "assistant", "content": s2_target(row["answer"])},
                ],
                "images": [str(ip.resolve())],
            }, ensure_ascii=True) + "\n")
            count += 1

    print(f"  s2_subset.jsonl: {count:,} rows written")
    _validate_and_fix(out)
    return out

def build_s2_test():
    out = JSONL_DIR / "s2_test.jsonl"
    if out.exists():
        lines = open(out, encoding="utf-8").readlines()
        bad   = sum(1 for l in lines if not _is_valid_json(l))
        if bad == 0 and len(lines) > 100:
            print(f"  s2_test.jsonl exists and valid: {len(lines):,} rows — skipping")
            return out
        print(f"  s2_test.jsonl corrupted — rebuilding")
        out.unlink()

    existing = {p.stem for p in IMG_DIR.glob("*.jpg")}
    count    = 0
    with open(out, "w", encoding="utf-8", newline="\n") as f:
        for row in tqdm(ds_x1["test"], desc="s2_test", leave=False):
            if row["img_id"] not in existing:
                continue
            f.write(json.dumps({
                "messages": [
                    {"role": "system",    "content": S2_SYS},
                    {"role": "user",      "content": f"<image>{row['question']}"},
                    {"role": "assistant", "content": s2_target(row["answer"])},
                ],
                "images": [str((IMG_DIR / f"{row['img_id']}.jpg").resolve())],
            }, ensure_ascii=True) + "\n")
            count += 1
    print(f"  s2_test.jsonl: {count:,} rows")
    _validate_and_fix(out)
    return out

# ── Build JSONLs ──────────────────────────────────────────────────────────────
print("\nBuilding S2 JSONLs ...")
s2_test = build_s2_test()
s2_curr = build_s2_subset(n_per_complexity=5000)   # 15K total

# ── Fixed output dir ──────────────────────────────────────────────────────────
out_root = CKPT_DIR / "medgemma" / "stage1" / "run1"
out_root.mkdir(parents=True, exist_ok=True)

# Verify Drive is writable
test_file = out_root / ".write_test"
try:
    test_file.write_text("ok")
    test_file.unlink()
    print(f"\n✅  Output dir writable: {out_root}")
except Exception as e:
    raise RuntimeError(f"Cannot write to output dir: {out_root}\n{e}")

# Check for checkpoint to resume from
existing_ckpts = sorted(
    [p for p in out_root.glob("checkpoint-*") if p.is_dir()],
    key=lambda p: int(p.name.split("-")[-1]),
)
resume_ckpt = existing_ckpts[-1] if existing_ckpts else None
if resume_ckpt:
    print(f"✅  Resuming from: {resume_ckpt.name}")
else:
    print("  No checkpoint found — training from scratch")

# ── Training environment ──────────────────────────────────────────────────────
def get_env():
    env = os.environ.copy()
    us  = site.getusersitepackages()
    env["PYTHONPATH"] = f"{us}:{env.get('PYTHONPATH', '')}"
    env.setdefault("TOKENIZERS_PARALLELISM", "false")
    env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    return env

# ── Training command ──────────────────────────────────────────────────────────
cmd = (
    f"{sys.executable} -m swift.cli.main sft"

    # MedGemma base — must be logged in to HF to access
    f" --model google/medgemma-4b-it"

    # Start from SimulaMet MedGemma adapter if available
    f" --adapters SimulaMet/MedGemma-KvasirVQA-x1-ft"

    # Data
    f" --dataset {s2_curr}"
    f" --val-dataset {s2_test}"
    f" --output-dir {out_root}"

    # Training
    f" --max-steps {MAX_STEPS}"
    f" --num-train-epochs 2"

    # Batch & optimiser
    f" --per-device-train-batch-size 1"
    f" --gradient-accumulation-steps 8"
    f" --learning-rate 5e-5"
    f" --lr-scheduler-type cosine"
    f" --warmup-ratio 0.03"

    # LoRA
    f" --lora-rank 8"
    f" --lora-alpha 16"
    f" --lora-dropout 0.05"
    f" --target-modules all-linear"

    # Quantisation — MedGemma needs fp16 not bf16
    f" --quant-bits 4"
    f" --bf16 false"
    f" --fp16 true"

    # Checkpointing — save every 200 steps
    f" --save-steps 200"
    f" --save-strategy steps"
    f" --eval-steps 400"
    f" --logging-steps 20"
    f" --save-total-limit 3"

    # Image cap + context
    f" --max-pixels 401408"
    f" --max-length {MAX_LEN}"

    # Stability
    f" --attn-impl sdpa"
    f" --use-hf true"
    f" --dataloader-num-workers 0"
    f" --gradient-checkpointing true"
)

# Add resume if checkpoint exists
if resume_ckpt:
    cmd += f" --resume-from-checkpoint {resume_ckpt}"

(out_root / "train.sh").write_text(f"#!/bin/bash\n{cmd}\n")

if DRY_RUN:
    print("\n[DRY RUN] Command:")
    for part in cmd.split(" --"):
        if part.strip():
            print(f"  --{part.strip()}"
                  if not part.startswith(sys.executable)
                  else f"  {part.strip()}")
    print(f"\nEstimated time: ~{MAX_STEPS * 4 / 3600:.1f} hrs on A100")

else:
    log = LOGS_DIR / "medgemma_stage1.log"
    print(f"\nTraining started.")
    print(f"  Steps     : {MAX_STEPS}  (~{MAX_STEPS*4/60:.0f} min on A100)")
    print(f"  Model     : google/medgemma-4b-it")
    print(f"  Adapter   : SimulaMet/MedGemma-KvasirVQA-x1-ft")
    print(f"  Dataset   : s2_subset.jsonl (15K rows, balanced L1/L2/L3)")
    print(f"  Output    : {out_root}")
    print(f"  Saves at  : every 200 steps")
    print(f"  Log       : {log}")
    if resume_ckpt:
        print(f"  Resuming  : {resume_ckpt.name}")
    print(f"  ⚠️  Requires HF login + MedGemma access approval")
    print(f"  (all startup lines shown, then loss/step/checkpoint)\n")

    t0         = time.time()
    line_count = 0

    proc = subprocess.Popen(
        cmd, shell=True, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env=get_env(), bufsize=1,
    )

    with open(log, "w") as lf:
        for line in proc.stdout:
            lf.write(line)
            lf.flush()
            s = line.rstrip()
            line_count += 1
            elapsed = (time.time() - t0) / 60

            if line_count <= 150:
                if s:
                    print(f"  [{elapsed:5.1f} min]  {s}")
                continue

            if any(k in s for k in [
                "'loss'", "loss", "train_loss", "eval_loss",
                "it/s", "s/it", "Train:", "Epoch",
                "Saving", "checkpoint", "saved",
                "Training completed",
                "Error", "error", "Traceback",
                "CUDA out", "401", "403", "gated", "Unauthorized",
            ]):
                print(f"  [{elapsed:6.1f} min]  {s}")

    proc.wait()
    rc      = proc.returncode
    elapsed = (time.time() - t0) / 60

    if rc == 0:
        ckpts = sorted(
            [p for p in out_root.glob("checkpoint-*") if p.is_dir()],
            key=lambda p: int(p.name.split("-")[-1]),
        )
        best = ckpts[-1] if ckpts else out_root
        print(f"\n✅  Training complete in {elapsed:.1f} min")
        print(f"    Checkpoints : {[c.name for c in ckpts]}")
        print(f"    Best        : {best}")
        print(f"    Next        : run Cell 09 (S2 inference)")
    else:
        print(f"\n❌  Failed (rc={rc}) after {elapsed:.1f} min")
        ckpts = sorted(
            [p for p in out_root.glob("checkpoint-*") if p.is_dir()],
            key=lambda p: int(p.name.split("-")[-1]),
        )
        if ckpts:
            print(f"    Partial checkpoints: {[c.name for c in ckpts]}")
            print(f"    Re-run to resume from: {ckpts[-1].name}")
        print(f"    Check log: {log}")
        print("    Last 30 lines:")
        try:
            for ln in open(log).readlines()[-30:]:
                print(f"      {ln.rstrip()}")
        except Exception:
            pass

DRY_RUN  = False
MAX_STEPS= 800

Building S2 JSONLs ...
  s2_test.jsonl corrupted — rebuilding


s2_test:   0%|          | 0/15955 [00:00<?, ?it/s]

  s2_test.jsonl: 15,955 rows
  ✅  All 15,955 lines valid JSON
  Loading rows for S2 subset ...
  Available: L1=54,856  L2=52,349  L3=52,344
  Sampled L1: 5,000
  Sampled L2: 5,000
  Sampled L3: 5,000
  Total: 15,000 rows


Writing s2_subset:   0%|          | 0/15000 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.
  [ 288.8 min]  Val: 100%|██████████| 15955/15955 [1:13:37<00:00,  3.61it/s]
  [ 288.9 min]  [INFO:swift] Saving model checkpoint to /content/drive/MyDrive/CSMORGAN_MEDVQA_2026/checkpoints/medgemma/stage1/run1/v0-20260510-235751/checkpoint-800
  [ 288.9 min]  Train: 100%|██████████| 800/800 [4:47:51<00:00, 21.59s/it]
  [ 288.9 min]  [INFO:swift] last_model_checkpoint: /content/drive/MyDrive/CSMORGAN_MEDVQA_2026/checkpoints/medgemma/stage1/run1/v0-20260510-235751/checkpoint-800
  [ 288.9 min]  [INFO:swift] best_model_checkpoint: /content/drive/MyDrive/CSMORGAN_MEDVQA_2026/checkpoints/medgemma/stage1/run1/v0-20260510-235751/checkpoint-800
  [ 288.9 min]  [INFO:swift] images_dir: /content/drive/MyDrive/CSMORGAN_MEDVQA_2026/checkpoints/medgemma/stage1/run1/v0-20260510-235751/images
  [ 288.9 min]  {'eval_loss': 0.13918526, 'eval_runtime': 4418.2049, 'eval_samples_per_second': 3.611, 'eval_steps_per_second': 3.611, 'eval_token_acc': 0.94910

In [ ]:
\
# ── CELL 06: Model Loading ────────────────────────────────────────────────────
from transformers import BitsAndBytesConfig
from swift.llm import PtEngine, RequestConfig

def bnb():
    return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                               bnb_4bit_use_double_quant=True,
                               bnb_4bit_compute_dtype=torch.float16)

def find_ckpt(model_key="qwen_transf"):
    """Find best local checkpoint. Falls back to HF adapter."""
    base = CKPT_DIR / model_key
    for stage in ["stage1", "stage1_fast"]:
        sd = base / stage
        if not sd.exists(): continue
        # Look inside v0-* subdirs (ms-swift output)
        for vdir in sorted(sd.glob("v0-*"), reverse=True):
            ckpts = sorted([c for c in vdir.glob("checkpoint-*") if c.is_dir()],
                           key=lambda c: int(c.name.split("-")[-1]))
            for c in reversed(ckpts):
                if (c/"adapter_model.safetensors").exists() or (c/"adapter_model.bin").exists():
                    return str(c)
        # Direct checkpoints
        ckpts = sorted([c for c in sd.glob("checkpoint-*") if c.is_dir()],
                       key=lambda c: int(c.name.split("-")[-1]))
        for c in reversed(ckpts):
            if (c/"adapter_model.safetensors").exists() or (c/"adapter_model.bin").exists():
                return str(c)
    return None

def load_engine(model_key="qwen_transf", local=True):
    cfg     = MODELS[model_key]
    adapter = (find_ckpt(model_key) if local else None) or cfg["adapter"]
    print(f"Loading {cfg['label']} from {adapter} ...")
    engine  = PtEngine(model_id_or_path=cfg["base"], adapters=[adapter],
                       quantization_config=bnb(), attn_impl="sdpa", use_hf=True)
    req_cfg = RequestConfig(max_tokens=64, temperature=0.1,
                            top_k=20, top_p=0.7, repetition_penalty=1.05)
    print(f"✅  {cfg['label']} ready")
    return engine, req_cfg

def unload(engine):
    del engine
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    gc.collect()
    print("  Engine unloaded")

# Quick test
engine, req_cfg = load_engine("qwen_transf", local=True)
from swift.llm import InferRequest
imgs = list(IMG_DIR.glob("*.jpg"))
if imgs:
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
        Image.open(imgs[0]).convert("RGB").save(tmp.name); tp = tmp.name
    try:
        resp = engine.infer([InferRequest(messages=[{"role":"user","content":[
            {"type":"image","image":tp},{"type":"text","text":"What is shown?"}
        ]}])], req_cfg)
        print("Test output:", repr(resp[0].choices[0].message.content[:80]))
    finally:
        try: os.unlink(tp)
        except: pass
unload(engine)


[INFO:swift] Downloading the model from HuggingFace Hub, model_id: Qwen/Qwen2.5-VL-7B-Instruct


Loading Qwen2.5-VL-7B-TransfFT from SimulaMet/Qwen2.5-VL-Transf-KvasirVQA-x1-ft ...


Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

[INFO:swift] Loading the model using model_dir: /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/cc594898137f460bfe9f0759e9844b3ce807cfb5
`torch_dtype` is deprecated! Use `dtype` instead!
[INFO:swift] Setting torch_dtype: torch.bfloat16
[WARNING:swift] Please install the package: `pip install "decord" -U`.
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
[INFO:swift] attn_impl: sdpa
[INFO:swift] model_kwargs: {'device_map': 'cuda:0', 'quantization_config': BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type"

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

...

training_args.bin:   0%|          | 0.00/7.80k [00:00<?, ?B/s]

[INFO:swift] Loading the model using model_dir: /root/.cache/huggingface/hub/models--SimulaMet--Qwen2.5-VL-Transf-KvasirVQA-x1-ft/snapshots/f241cfd234615b1e8a8583923ad050a958549aa2
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:585: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.visual.blocks.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.visual.blocks.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.visual.blocks.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.visual.blocks.0.mlp.up_proj.lora_B.default.weight', 'base_model.model.model.visual.blocks.0.mlp.down_proj.lora_A.default.weight', 'base_model.model.model.visual.blocks.0.mlp.down_proj.lora_B.default.weight', 'base_model.model.model.visual.blocks.1.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.visual.blocks.1.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.visual.blocks.1.mlp.up_proj.lora_A.defau

✅  Qwen2.5-VL-7B-TransfFT ready


[INFO:swift] Successfully registered `/usr/local/lib/python3.12/dist-packages/swift/llm/dataset/data/dataset_info.json`.


Test output: 'The image appears to be a colonoscopy view, which is a medical procedure used to'
  Engine unloaded


In [ ]:
# ── CELL 09: Task 2 Inference — MedGemma via swift ───────────────────────────
import gc, json, os, re, time, tempfile
import torch
from PIL import Image
from tqdm.auto import tqdm
from swift.llm import PtEngine, RequestConfig, InferRequest
from transformers import BitsAndBytesConfig

# ============================================================
# Config
# ============================================================
RESUME      = False         # False for fresh debug run
MAX_SAMPLES = 1000           # Use None for full test set after checking quality
MG_BASE     = "google/medgemma-4b-it"
MG_HF       = "SimulaMet/MedGemma-KvasirVQA-x1-ft"

BAD_ANSWERS = {
    "",
    "unknown",
    "none",
    "null",
    "nan",
    "n/a",
    "na",
    "not available",
    "no answer",
    "unanswered",
}

GENERIC_JUSTIFICATIONS = {
    "based on the visible endoscopic findings in the image.",
    "based on visible endoscopic findings.",
    "the answer is based on visible endoscopic findings.",
    "based on the image.",
    "based on the visible findings.",
}

# ============================================================
# Find latest MedGemma checkpoint
# ============================================================
def find_mg_ckpt():
    base = CKPT_DIR / "medgemma" / "stage1" / "run1"

    if not base.exists():
        base = CKPT_DIR / "medgemma" / "stage1"

    if not base.exists():
        return None

    version_dirs = sorted(base.glob("v*"), reverse=True)

    for vdir in version_dirs:
        ckpts = sorted(
            [c for c in vdir.glob("checkpoint-*") if c.is_dir()],
            key=lambda c: int(c.name.split("-")[-1]),
        )

        for c in reversed(ckpts):
            has_adapter = (
                (c / "adapter_model.safetensors").exists()
                or (c / "adapter_model.bin").exists()
            )

            if has_adapter:
                print(f"  Found: {c}")
                return str(c)

    return None


adapter = find_mg_ckpt() or MG_HF

print(f"Adapter    : {adapter}")
print(f"All samples: {len(test_samples):,}")

# ============================================================
# Load S1 predictions
# ============================================================
s1_json = RESULTS_DIR / "s1_predictions.json"

if s1_json.exists():
    s1_preds = {
        str(r["img_id"]): str(r.get("prediction", "")).strip()
        for r in json.load(open(s1_json, "r", encoding="utf-8"))
    }
    print(f"S1 loaded  : {len(s1_preds):,}")
else:
    s1_preds = {}
    print("⚠️  No S1 predictions found")


# ============================================================
# Helpers
# ============================================================
def clean_answer(ans):
    """
    Cleans answer text.
    Prevents weak S1 answers like 'unknown' from overriding MedGemma output.
    """
    ans = str(ans).strip()

    # Remove accidental field label
    ans = re.sub(r"^ANSWER:\s*", "", ans, flags=re.I).strip()

    # Keep only first line for answer
    ans = ans.split("\n")[0].strip()

    # Remove surrounding quotes/backticks
    ans = ans.strip("'\"` ")

    if ans.lower() in BAD_ANSWERS:
        return "unknown"

    return ans


def is_good_answer(ans):
    ans = clean_answer(ans)
    return ans.lower() not in BAD_ANSWERS and ans.lower() != "unknown"


def is_generic_justification(text):
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text in GENERIC_JUSTIFICATIONS or len(text) < 20


def improve_generic_justification(answer, question):
    """
    Fallback when the model gives a vague explanation.
    This does not invent image findings. It states that the model answer was based
    on visible review but did not localize enough evidence.
    """
    answer = clean_answer(answer)
    q = str(question).lower()

    if "text" in q:
        return (
            f"The model answered '{answer}' after checking the image for visible written "
            "or overlaid text. No specific localized text region was described by the model."
        )

    if "instrument" in q or "procedure" in q or "tube" in q:
        return (
            f"The model answered '{answer}' after checking the image for visible endoscopic "
            "tools or procedural devices. The generated explanation did not provide enough "
            "localized detail about instrument position or appearance."
        )

    if "polyp" in q:
        return (
            f"The model answered '{answer}' after checking the image for polypoid lesions. "
            "The generated explanation did not provide enough localized detail about lesion "
            "morphology, colour, size, or position."
        )

    if "artefact" in q or "artifact" in q or "box" in q:
        return (
            f"The model answered '{answer}' after checking the image for visual artefacts "
            "such as green or black box regions. The generated explanation did not provide "
            "a more specific localized artefact description."
        )

    if "landmark" in q or "z-line" in q or "anatomical" in q:
        return (
            f"The model answered '{answer}' after checking for visible anatomical landmarks. "
            "The generated explanation did not provide enough localized landmark detail."
        )

    return (
        f"The model answered '{answer}' after reviewing the endoscopic image. "
        "However, the generated explanation was generic and did not provide enough "
        "specific localized visual evidence."
    )


def parse_s2(text):
    """
    Parses structured MedGemma output:
    ANSWER:
    JUSTIFICATION:
    CONFIDENCE:
    SAFETY_NOTE:
    """
    out = {
        "ANSWER": "",
        "JUSTIFICATION": "",
        "CONFIDENCE": 0.5,
        "SAFETY_NOTE": "None",
    }

    text = str(text).strip()

    patterns = {
        "ANSWER": r"ANSWER:\s*(.*?)(?=\n\s*JUSTIFICATION:|\n\s*CONFIDENCE:|\n\s*SAFETY_NOTE:|$)",
        "JUSTIFICATION": r"JUSTIFICATION:\s*(.*?)(?=\n\s*CONFIDENCE:|\n\s*SAFETY_NOTE:|$)",
        "CONFIDENCE": r"CONFIDENCE:\s*([0-9]*\.?[0-9]+)",
        "SAFETY_NOTE": r"SAFETY_NOTE:\s*(.*?)(?=$)",
    }

    for key, pat in patterns.items():
        m = re.search(pat, text, flags=re.I | re.DOTALL)
        if m:
            out[key] = m.group(1).strip()

    out["ANSWER"] = clean_answer(out["ANSWER"])

    try:
        conf = float(out["CONFIDENCE"])
        out["CONFIDENCE"] = max(0.0, min(conf, 0.85))
    except Exception:
        out["CONFIDENCE"] = 0.5

    if not out["SAFETY_NOTE"]:
        out["SAFETY_NOTE"] = "None"

    return out


def build_prompt(question, s1_hint):
    """
    Builds Task 2 prompt.
    Does not force MedGemma to justify bad S1 answers like 'unknown'.
    """
    question = str(question).strip()
    s1_hint = clean_answer(s1_hint)

    prompt_txt = question

    if is_good_answer(s1_hint):
        prompt_txt += (
            f"\n\nA previous model suggested this answer: '{s1_hint}'. "
            "Verify this answer against the image. If it is wrong, correct it. "
            "Then explain the final answer using visible image evidence."
        )
    else:
        prompt_txt += (
            "\n\nAnswer directly from the image. Then justify your answer using "
            "visible image evidence."
        )

    prompt_txt += (
        "\n\nYour JUSTIFICATION must be 2 to 3 sentences. "
        "Mention specific visual details when visible, such as location, colour, "
        "morphology, size, texture, instruments, artefacts, anatomical landmarks, "
        "or absence of findings. Do not use generic phrases like "
        "'Based on the visible endoscopic findings in the image.'"
    )

    return prompt_txt


def choose_final_answer(s1_answer, mg_answer):
    """
    Uses S1 only when meaningful.
    Otherwise falls back to MedGemma.
    """
    s1_answer = clean_answer(s1_answer)
    mg_answer = clean_answer(mg_answer)

    if is_good_answer(s1_answer):
        return s1_answer

    if is_good_answer(mg_answer):
        return mg_answer

    return "unknown"


# ============================================================
# Resume handling
# ============================================================
out_path = RESULTS_DIR / "s2_results.json"
s2_results = []
done_ids = set()

if RESUME and out_path.exists():
    existing = json.load(open(out_path, "r", encoding="utf-8"))

    for r in existing:
        raw_ok = str(r.get("_raw", "")).strip()
        ans_ok = is_good_answer(r.get("answer", ""))

        if raw_ok or ans_ok:
            s2_results.append(r)
            done_ids.add(str(r["img_id"]))

    print(f"Resuming   : {len(done_ids):,} valid done")

remaining = [
    r for r in test_samples
    if str(r["img_id"]) not in done_ids
    and (IMG_DIR / f"{r['img_id']}.jpg").exists()
]

if MAX_SAMPLES is not None:
    remaining = remaining[:MAX_SAMPLES]

print(f"Running    : {len(remaining):,}")


# ============================================================
# Inference
# ============================================================
if not remaining:
    print("✅ Done. No remaining samples.")
else:
    print("Loading MedGemma via swift ...")

    engine = PtEngine(
        model_id_or_path=MG_BASE,
        adapters=[adapter],
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        ),
        attn_impl="sdpa",
        use_hf=True,
        max_length=2048,
    )

    req_cfg = RequestConfig(
        max_tokens=260,
        temperature=0.15,
        top_k=20,
        top_p=0.8,
        repetition_penalty=1.08,
    )

    print("✅ MedGemma loaded")

    S2_SYS = (
        "You are a clinically trained GI endoscopy AI. "
        "Look carefully at the image and answer the question.\n\n"
        "Return exactly this format:\n"
        "ANSWER: <short direct answer>\n"
        "JUSTIFICATION: <2-3 sentences describing specific visible evidence from the image. "
        "Mention location, colour, morphology, size, texture, instruments, artefacts, "
        "anatomical landmarks, or absence of findings when relevant. Do not write generic "
        "phrases like 'Based on the visible endoscopic findings in the image.'>\n"
        "CONFIDENCE: <float from 0.0 to 0.85>\n"
        "SAFETY_NOTE: <clinical concern or None>\n\n"
        "Rules:\n"
        "1. Do not invent findings.\n"
        "2. If no polyp, instrument, text, artefact, or landmark is visible, explicitly say what is absent.\n"
        "3. The justification must explain the answer using image-visible evidence.\n"
        "4. Avoid vague explanations.\n"
        "5. Keep the answer short, but make the justification clinically useful."
    )

    SAVE_EVERY = 25
    t0 = time.time()

    for idx, row in enumerate(tqdm(remaining, desc="S2")):
        img_id = str(row["img_id"])
        img_path = IMG_DIR / f"{img_id}.jpg"

        raw = ""

        try:
            img = Image.open(img_path).convert("RGB")
            img.thumbnail((448, 448), Image.LANCZOS)

            with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
                img.save(tmp.name, quality=95)
                tp = tmp.name

            s1_hint = s1_preds.get(img_id, "")
            prompt_txt = build_prompt(row["question"], s1_hint)

            try:
                resp = engine.infer(
                    [
                        InferRequest(
                            messages=[
                                {"role": "system", "content": S2_SYS},
                                {
                                    "role": "user",
                                    "content": [
                                        {"type": "image", "image": tp},
                                        {"type": "text", "text": prompt_txt},
                                    ],
                                },
                            ]
                        )
                    ],
                    req_cfg,
                )

                raw = resp[0].choices[0].message.content.strip()

            except Exception as e:
                raw = ""
                print(f"\n⚠️ Inference failed for img_id={img_id}: {repr(e)}")

            finally:
                try:
                    os.unlink(tp)
                except OSError:
                    pass

        except Exception as e:
            raw = ""
            print(f"\n⚠️ Image load failed for img_id={img_id}: {repr(e)}")

        fields = parse_s2(raw)

        mg_answer = fields["ANSWER"]
        s1_answer = s1_preds.get(img_id, "")

        answer = choose_final_answer(s1_answer, mg_answer)

        justif = str(fields.get("JUSTIFICATION", "")).strip()

        if is_generic_justification(justif):
            justif = improve_generic_justification(answer, row["question"])

        confidence = fields["CONFIDENCE"]

        result = {
            "val_id": len(s2_results),
            "img_id": img_id,
            "question": row["question"],
            "answer": answer,
            "textual_explanation": justif,
            "visual_explanation": [],
            "confidence_score": confidence,
            "_safety_note": fields["SAFETY_NOTE"],
            "_s1_answer": clean_answer(s1_answer),
            "_mg_answer": clean_answer(mg_answer),
            "_raw": raw,
        }

        s2_results.append(result)

        if len(s2_results) % SAVE_EVERY == 0:
            with open(out_path, "w", encoding="utf-8") as f:
                json.dump(s2_results, f, indent=2, ensure_ascii=False)

        # Monitor first few samples
        if idx < 10:
            print(f"\n  [{idx}] Q  : {str(row['question'])[:80]}")
            print(f"       S1 : {clean_answer(s1_answer)}")
            print(f"       MG : {clean_answer(mg_answer)}")
            print(f"       A  : {answer}")
            print(f"       raw: {repr(raw[:250])}")
            print(f"       J  : {justif[:180]}...")
            print(f"       C  : {confidence}")

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(s2_results, f, indent=2, ensure_ascii=False)

    del engine
    torch.cuda.empty_cache()
    gc.collect()

    elapsed = (time.time() - t0) / 60
    print(f"\nEngine unloaded. Time: {elapsed:.1f} min")


# ============================================================
# Save official JSONL
# ============================================================
jsonl_out = RESULTS_DIR / "submission_task2.jsonl"

with open(jsonl_out, "w", encoding="utf-8") as f:
    for i, r in enumerate(s2_results):
        f.write(
            json.dumps(
                {
                    "val_id": i,
                    "img_id": r["img_id"],
                    "question": r["question"],
                    "answer": r["answer"],
                    "textual_explanation": r["textual_explanation"],
                    "visual_explanation": r["visual_explanation"],
                    "confidence_score": r["confidence_score"],
                },
                ensure_ascii=False,
            )
            + "\n"
        )


# ============================================================
# Validate official JSONL
# ============================================================
required = [
    "val_id",
    "img_id",
    "question",
    "answer",
    "textual_explanation",
    "visual_explanation",
    "confidence_score",
]

errors = []

with open(jsonl_out, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        try:
            obj = json.loads(line)
        except Exception as e:
            errors.append(f"line {i}: invalid JSON: {repr(e)}")
            continue

        missing = [k for k in required if k not in obj]

        if missing:
            errors.append(f"line {i}: missing {missing}")

        if not isinstance(obj["visual_explanation"], list):
            errors.append(f"line {i}: visual_explanation must be a list")

        try:
            c = float(obj["confidence_score"])
            if not (0.0 <= c <= 1.0):
                errors.append(f"line {i}: confidence_score outside [0,1]")
        except Exception:
            errors.append(f"line {i}: confidence_score is not numeric")

n = sum(1 for _ in open(jsonl_out, "r", encoding="utf-8"))

if len(s2_results) > 0:
    mean_c = sum(float(r["confidence_score"]) for r in s2_results) / len(s2_results)
    empty_raw = sum(1 for r in s2_results if not str(r.get("_raw", "")).strip())
    unknown_ans = sum(
        1 for r in s2_results
        if clean_answer(r.get("answer", "")).lower() == "unknown"
    )
    generic_j = sum(
        1 for r in s2_results
        if is_generic_justification(r.get("textual_explanation", ""))
    )
else:
    mean_c = 0.0
    empty_raw = 0
    unknown_ans = 0
    generic_j = 0

print(f"\n✅ submission_task2.jsonl: {n:,} lines")
print(f"   Mean confidence       : {mean_c:.3f}")
print(f"   Empty raw             : {empty_raw}/{len(s2_results)}")
print(f"   Unknown answers       : {unknown_ans}/{len(s2_results)}")
print(f"   Generic explanations  : {generic_j}/{len(s2_results)}")

if errors:
    print(f"   ⚠️ {len(errors)} validation errors")
    for e in errors[:20]:
        print("   -", e)
else:
    print("   ✅ All fields valid")


# ============================================================
# Show sample outputs
# ============================================================
print("\nSample outputs:")
for r in s2_results[:5]:
    print(f"  Q : {str(r['question'])[:80]}")
    print(f"  S1: {r.get('_s1_answer', '')}")
    print(f"  MG: {r.get('_mg_answer', '')}")
    print(f"  A : {r['answer']}")
    print(f"  J : {str(r['textual_explanation'])[:160]}...")
    print(f"  C : {r['confidence_score']}")
    print()

print("Next: run Cell 10 to build ZIP")

Adapter    : SimulaMet/MedGemma-KvasirVQA-x1-ft
All samples: 15,955
S1 loaded  : 2,835


[INFO:swift] Downloading the model from HuggingFace Hub, model_id: google/medgemma-4b-it


Running    : 1,000
Loading MedGemma via swift ...


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

[INFO:swift] Loading the model using model_dir: /root/.cache/huggingface/hub/models--google--medgemma-4b-it/snapshots/290cda5eeccbee130f987c4ad74a59ae6f196408
[INFO:swift] Setting model_type: gemma3_vision
[INFO:swift] Setting torch_dtype: torch.bfloat16
[INFO:swift] attn_impl: sdpa
[INFO:swift] model_kwargs: {'device_map': 'cuda:0', 'quantization_config': BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "nf4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}
}


...

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

[INFO:swift] Loading the model using model_dir: /root/.cache/huggingface/hub/models--SimulaMet--MedGemma-KvasirVQA-x1-ft/snapshots/294353a9ea33a90e07862a23b0b26a82dcc6164d
[INFO:swift] Create the default_template for the infer_engine
[INFO:swift] default_system: None
[INFO:swift] max_length: 131072
[INFO:swift] response_prefix: ''
[INFO:swift] agent_template: react_en
[INFO:swift] norm_bbox: norm1000


✅ MedGemma loaded


S2:   0%|          | 0/1000 [00:00<?, ?it/s]


  [0] Q  : Have all identified polyps been successfully removed?
       S1 : Multiple polyps observed scattered across central and upper regions of the image.
       MG : Not all polyps have been removed
       A  : Multiple polyps observed scattered across central and upper regions of the image.
       raw: 'ANSWER: Not all polyps have been removed\nJUSTIFICATION: Multiple polyps are present in the central and upper regions of the image.\nCONFIDENCE: 0.95\nSAFETY_NOTE: No significant abnormalities detected'
       J  : Multiple polyps are present in the central and upper regions of the image....
       C  : 0.85

  [1] Q  : How many polyps are present, what colors are the abnormalities, and is there any
       S1 : No residual polyps identified, text visible
       MG : No polyps observed, multiple colored abnormalities noted, no instrument visible.
       A  : No residual polyps identified, text visible
       raw: 'ANSWER: No polyps observed, multiple colored abnormalities noted, n

In [ ]:
# ── CELL 09aa: Materialize Missing Task 2 Images into IMG_DIR ────────────────
# This cell tries to save all missing Task 2 images into IMG_DIR so Cell 09b
# can generate SAM visual explanations.
#
# It searches:
#   1. Existing notebook dataset variables
#   2. test_samples if images are embedded there
#   3. HuggingFace dataset fallback
#
# After this cell, rerun Cell 09b.

import json
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm

IMG_DIR = Path(IMG_DIR)
IMG_DIR.mkdir(parents=True, exist_ok=True)

s2_json_path = RESULTS_DIR / "s2_results.json"
assert s2_json_path.exists(), "Run Cell 09 first to create s2_results.json"

with open(s2_json_path, "r", encoding="utf-8") as f:
    s2_results = json.load(f)

needed_ids = [str(r["img_id"]) for r in s2_results]
needed_set = set(needed_ids)

print("Needed Task 2 images:", len(needed_set))
print("IMG_DIR:", IMG_DIR)

# ---------------------------------------------------------------------
# Existing image index
# ---------------------------------------------------------------------
IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

def build_existing_index(root):
    idx = {}
    for p in Path(root).rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            idx[p.stem] = p
    return idx

existing = build_existing_index(IMG_DIR)
missing_ids = sorted([x for x in needed_set if x not in existing])

print("Existing images in IMG_DIR:", len(existing))
print("Missing Task 2 images:", len(missing_ids))

if len(missing_ids) == 0:
    print("✅ All Task 2 images already exist.")
else:
    print("First 10 missing IDs:")
    for x in missing_ids[:10]:
        print(" ", x)

# ---------------------------------------------------------------------
# Helper: save PIL image
# ---------------------------------------------------------------------
def save_pil_image(img, img_id):
    if img is None:
        return False

    try:
        if not isinstance(img, Image.Image):
            return False

        img = img.convert("RGB")
        out_path = IMG_DIR / f"{img_id}.jpg"
        img.save(out_path, quality=95)
        return True
    except Exception as e:
        print(f"Failed saving {img_id}: {repr(e)}")
        return False

# ---------------------------------------------------------------------
# Helper: get ID from dataset row
# ---------------------------------------------------------------------
def get_img_id_from_row(row):
    for key in ["img_id", "image_id", "id", "image_name", "filename"]:
        if key in row:
            v = str(row[key])
            return Path(v).stem
    return None

def get_image_from_row(row):
    for key in ["image", "img", "image_file"]:
        if key in row and isinstance(row[key], Image.Image):
            return row[key]

    # path-style fields
    for key in ["image_path", "path", "filepath", "file_name", "filename"]:
        if key in row:
            p = Path(str(row[key]))
            if p.exists():
                try:
                    return Image.open(p).convert("RGB")
                except Exception:
                    pass

    return None

# ---------------------------------------------------------------------
# Source 1: test_samples if it contains images
# ---------------------------------------------------------------------
saved = 0

if "test_samples" in globals():
    print("\nSearching test_samples for embedded images...")
    for row in tqdm(test_samples, desc="test_samples"):
        img_id = str(row.get("img_id", ""))
        if img_id not in missing_ids:
            continue

        img = get_image_from_row(row)
        if save_pil_image(img, img_id):
            saved += 1

    print("Saved from test_samples:", saved)

# Refresh missing
existing = build_existing_index(IMG_DIR)
missing_ids = sorted([x for x in needed_set if x not in existing])
print("Still missing after test_samples:", len(missing_ids))

# ---------------------------------------------------------------------
# Source 2: search common dataset variables already loaded in notebook
# ---------------------------------------------------------------------
candidate_var_names = [
    "val_dataset",
    "test_dataset",
    "dataset",
    "ds",
    "task2_dataset",
    "task2_val_dataset",
    "val_ds",
    "test_ds",
]

saved_from_vars = 0

for var_name in candidate_var_names:
    if len(missing_ids) == 0:
        break

    if var_name not in globals():
        continue

    obj = globals()[var_name]
    print(f"\nSearching notebook variable: {var_name}")

    try:
        for row in tqdm(obj, desc=var_name):
            img_id = get_img_id_from_row(row)
            if img_id is None or img_id not in missing_ids:
                continue

            img = get_image_from_row(row)
            if save_pil_image(img, img_id):
                saved_from_vars += 1

        existing = build_existing_index(IMG_DIR)
        missing_ids = sorted([x for x in needed_set if x not in existing])
        print(f"Still missing after {var_name}:", len(missing_ids))

    except Exception as e:
        print(f"Could not search {var_name}: {repr(e)}")

print("Saved from loaded variables:", saved_from_vars)

# ---------------------------------------------------------------------
# Source 3: HuggingFace fallback
# ---------------------------------------------------------------------
existing = build_existing_index(IMG_DIR)
missing_ids = sorted([x for x in needed_set if x not in existing])

if len(missing_ids) > 0:
    print("\nTrying HuggingFace fallback dataset loading...")

    try:
        from datasets import load_dataset

        # Try likely dataset names.
        # The first one is the one your Task 1 script used.
        hf_candidates = [
            ("SimulaMet/Kvasir-VQA-test", "validation"),
            ("SimulaMet/Kvasir-VQA-x1", "validation"),
            ("SimulaMet/Kvasir-VQA-x1", "test"),
        ]

        saved_hf = 0

        for ds_name, split in hf_candidates:
            if len(missing_ids) == 0:
                break

            print(f"\nLoading {ds_name} split={split} ...")

            try:
                ds = load_dataset(ds_name, split=split)
            except Exception as e:
                print(f"Could not load {ds_name}/{split}: {repr(e)}")
                continue

            print("Dataset rows:", len(ds))
            print("Columns:", ds.column_names)

            for row in tqdm(ds, desc=f"{ds_name}/{split}"):
                img_id = get_img_id_from_row(row)

                if img_id is None or img_id not in missing_ids:
                    continue

                img = get_image_from_row(row)

                if save_pil_image(img, img_id):
                    saved_hf += 1

            existing = build_existing_index(IMG_DIR)
            missing_ids = sorted([x for x in needed_set if x not in existing])
            print(f"Still missing after {ds_name}/{split}:", len(missing_ids))

        print("Saved from HuggingFace:", saved_hf)

    except Exception as e:
        print("HF fallback failed:", repr(e))

# ---------------------------------------------------------------------
# Final report
# ---------------------------------------------------------------------
existing = build_existing_index(IMG_DIR)
available_needed = [x for x in needed_set if x in existing]
still_missing = sorted([x for x in needed_set if x not in existing])

print("\n" + "=" * 70)
print("IMAGE MATERIALIZATION REPORT")
print("=" * 70)
print("Needed Task 2 images :", len(needed_set))
print("Available now        :", len(available_needed))
print("Still missing        :", len(still_missing))
print("IMG_DIR total images :", len(existing))

if still_missing:
    print("\nFirst 20 still missing:")
    for x in still_missing[:20]:
        print(" ", x)
else:
    print("\n✅ All Task 2 images are now available.")

print("=" * 70)

Needed Task 2 images: 750
IMG_DIR: /content/drive/MyDrive/CSMORGAN_MEDVQA_2026/data/images
Existing images in IMG_DIR: 6500
Missing Task 2 images: 0
✅ All Task 2 images already exist.

Searching test_samples for embedded images...


test_samples:   0%|          | 0/15955 [00:00<?, ?it/s]

Saved from test_samples: 0
Still missing after test_samples: 0
Saved from loaded variables: 0

IMAGE MATERIALIZATION REPORT
Needed Task 2 images : 750
Available now        : 750
Still missing        : 0
IMG_DIR total images : 6500

✅ All Task 2 images are now available.


In [ ]:
# ── CELL 09b: Generate Task 2 Visual Explanations with SAM-Kvasir ─────────────
# Fixed version:
#   - Robustly finds images with .jpg / .jpeg / .png / .bmp / .webp
#   - Searches recursively under IMG_DIR
#   - Uses SAM-Kvasir for segmentation masks
#   - Saves mask PNG, overlay PNG, and bbox JSON
#   - Updates submission_task2.jsonl with visual_explanation entries
#
# Run after:
#   Cell 09
#   Cell 09a

import os
import json
import cv2
import numpy as np
from pathlib import Path
from PIL import Image

import torch
from transformers import SamModel, SamProcessor
from tqdm.auto import tqdm

# ============================================================
# CONFIG
# ============================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

VIS_DIR = RESULTS_DIR / "visuals"
VIS_DIR.mkdir(parents=True, exist_ok=True)

s2_json_path = RESULTS_DIR / "s2_results.json"
jsonl_path   = RESULTS_DIR / "submission_task2.jsonl"

assert s2_json_path.exists(), "Run Cell 09 first to generate s2_results.json"
assert jsonl_path.exists(), "Run Cell 09 first to generate submission_task2.jsonl"

# Pretrained SAM model fine-tuned on Kvasir-SEG
SAM_POLYP_MODEL_ID = "Mayank022/sam-vit-base-kvasir-polyp-segmentation"

# Use None for all entries.
# Use 50 or 100 for debugging.
MAX_VISUAL_SAMPLES = None
# MAX_VISUAL_SAMPLES = 50

# If True, deletes old visual files before generating new ones.
DELETE_OLD_VISUALS = True

# ============================================================
# Optional: clean old visuals
# ============================================================
if DELETE_OLD_VISUALS and VIS_DIR.exists():
    deleted = 0
    for p in VIS_DIR.rglob("*"):
        if p.is_file():
            p.unlink()
            deleted += 1
    print(f"Deleted old visual files: {deleted}")

# ============================================================
# Debug paths
# ============================================================
print("IMG_DIR     :", IMG_DIR)
print("IMG_DIR ok  :", Path(IMG_DIR).exists())
print("RESULTS_DIR :", RESULTS_DIR)
print("VIS_DIR     :", VIS_DIR)
print("Device      :", DEVICE)

# ============================================================
# Robust image path resolver
# ============================================================
IMAGE_EXTS = [
    ".jpg", ".jpeg", ".png", ".bmp", ".webp",
    ".JPG", ".JPEG", ".PNG", ".BMP", ".WEBP",
]

print("\nIndexing image files under IMG_DIR...")

IMG_INDEX = {}
img_root = Path(IMG_DIR)

if img_root.exists():
    for p in img_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
            # Use stem because img_id usually excludes extension
            IMG_INDEX[p.stem] = p

print(f"Indexed image files: {len(IMG_INDEX):,}")

if len(IMG_INDEX) > 0:
    print("Sample indexed image names:")
    for k, v in list(IMG_INDEX.items())[:10]:
        print(" ", k, "->", v.name)
else:
    print("⚠️ No images found under IMG_DIR. Check IMG_DIR path.")

def find_image_path(img_id):
    """
    Finds image path by:
      1. direct IMG_DIR/img_id.ext
      2. recursive indexed stem lookup
      3. raw IMG_DIR/img_id if img_id already has extension
    """
    img_id = str(img_id).strip()

    # If img_id already has extension
    p = Path(IMG_DIR) / img_id
    if p.exists() and p.is_file():
        return p

    # Direct checks
    for ext in IMAGE_EXTS:
        p = Path(IMG_DIR) / f"{img_id}{ext}"
        if p.exists():
            return p

    # Recursive index by stem
    if img_id in IMG_INDEX:
        return IMG_INDEX[img_id]

    return None

# ============================================================
# Trigger logic
# ============================================================
TRIGGER_WORDS = [
    "polyp", "polyps",
    "lesion", "lesions",
    "abnormal", "abnormality", "abnormalities",
    "inflammation", "inflamed",
    "ulcer", "ulcerative", "erosion",
    "mass", "finding", "findings",
    "instrument", "forceps", "tube", "device",
    "z-line", "landmark",
    "artifact", "artefact",
    "colitis", "esophagitis",
    "bleeding", "red", "pink", "white"
]

NEGATIVE_PHRASES = [
    "no polyp",
    "no polyps",
    "no polypoid",
    "no lesion",
    "no lesions",
    "no abnormal",
    "no significant abnormal",
    "no visible abnormal",
    "no instrument",
    "no instruments",
    "no visible instrumentation",
    "no surgical instruments",
    "no text",
    "no visible text",
    "no evidence",
    "not visible",
    "not observed",
    "not identified",
    "none identified",
    "no anatomical landmarks",
]


def should_generate_visuals(answer, explanation, question):
    """
    Avoid fake localization for clearly negative answers.
    Generate visuals only for likely visible target regions.
    """
    txt = f"{question} {answer} {explanation}".lower()

    if any(p in txt for p in NEGATIVE_PHRASES):
        return False

    return any(w in txt for w in TRIGGER_WORDS)

# ============================================================
# Utility functions
# ============================================================
def read_bgr(img_path):
    img = cv2.imread(str(img_path))
    if img is None:
        raise ValueError(f"Could not read image: {img_path}")
    return img


def save_json(path, obj):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


def bbox_from_mask(mask_u8, min_area_ratio=0.001):
    """
    mask_u8: binary mask 0/255
    returns [x1, y1, x2, y2] or None
    """
    h, w = mask_u8.shape[:2]

    contours, _ = cv2.findContours(
        mask_u8,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE,
    )

    if not contours:
        return None

    c = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(c)

    if area < min_area_ratio * h * w:
        return None

    x, y, ww, hh = cv2.boundingRect(c)

    return [
        int(x),
        int(y),
        int(x + ww),
        int(y + hh),
    ]


def make_mask_overlay(img_bgr, mask_u8, bbox=None):
    """
    Creates a green mask overlay and optional yellow bounding box.
    """
    overlay = img_bgr.copy()

    color_mask = np.zeros_like(img_bgr)
    color_mask[:, :, 1] = mask_u8

    overlay = cv2.addWeighted(overlay, 0.70, color_mask, 0.30, 0)

    contours, _ = cv2.findContours(
        mask_u8,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE,
    )

    if contours:
        cv2.drawContours(overlay, contours, -1, (0, 255, 0), 2)

    if bbox is not None:
        x1, y1, x2, y2 = bbox
        cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 255), 2)

    return overlay


# ============================================================
# Heuristic prompt box for SAM
# ============================================================
def heuristic_bbox_prompt(img_bgr):
    """
    Finds a rough candidate ROI to prompt SAM.
    This is only the prompt box. SAM produces the actual mask.
    """
    h, w = img_bgr.shape[:2]

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    tissue_mask = gray > 20

    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)

    b, g, r = cv2.split(img_bgr)

    redness = (
        r.astype(np.float32)
        - 0.50 * g.astype(np.float32)
        - 0.25 * b.astype(np.float32)
    )
    redness = np.clip(redness, 0, None)

    l_chan, _, _ = cv2.split(lab)
    brightness = l_chan.astype(np.float32)
    saturation = hsv[:, :, 1].astype(np.float32)

    score = (
        0.55 * cv2.GaussianBlur(redness, (0, 0), 5)
        + 0.20 * cv2.GaussianBlur(brightness, (0, 0), 7)
        + 0.25 * cv2.GaussianBlur(saturation, (0, 0), 5)
    )

    score[~tissue_mask] = 0

    if score.max() <= score.min():
        return None

    score = (score - score.min()) / (score.max() - score.min())
    heat = (score * 255).astype(np.uint8)

    positive = heat[heat > 0]
    if len(positive) == 0:
        return None

    thresh_val = max(40, int(np.percentile(positive, 85)))
    _, bw = cv2.threshold(heat, thresh_val, 255, cv2.THRESH_BINARY)

    kernel = np.ones((7, 7), np.uint8)
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, kernel)
    bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, kernel)

    bbox = bbox_from_mask(bw, min_area_ratio=0.002)

    if bbox is None:
        # fallback center box
        margin_x = int(w * 0.20)
        margin_y = int(h * 0.20)
        bbox = [
            margin_x,
            margin_y,
            w - margin_x,
            h - margin_y,
        ]

    # Expand box slightly for SAM
    x1, y1, x2, y2 = bbox
    box_w = max(1, x2 - x1)
    box_h = max(1, y2 - y1)

    pad_x = int(0.08 * box_w)
    pad_y = int(0.08 * box_h)

    x1 = max(0, x1 - pad_x)
    y1 = max(0, y1 - pad_y)
    x2 = min(w - 1, x2 + pad_x)
    y2 = min(h - 1, y2 + pad_y)

    return [
        int(x1),
        int(y1),
        int(x2),
        int(y2),
    ]

# ============================================================
# Load SAM-Kvasir
# ============================================================
print("\nLoading SAM-Kvasir polyp segmentation model...")
print("Model:", SAM_POLYP_MODEL_ID)

sam_processor = SamProcessor.from_pretrained(SAM_POLYP_MODEL_ID)
sam_model = SamModel.from_pretrained(SAM_POLYP_MODEL_ID).to(DEVICE)
sam_model.eval()

print("✅ SAM-Kvasir loaded")
print("Device:", DEVICE)

# ============================================================
# Run SAM segmentation
# ============================================================
def run_sam_polyp_segmentation(img_path, img_id):
    """
    Saves:
      - visuals/<img_id>_sam_mask.png
      - visuals/<img_id>_sam_overlay.png
      - visuals/<img_id>_sam_bbox.json

    Returns visual_explanation list.
    """
    visual_items = []

    try:
        pil_img = Image.open(img_path).convert("RGB")
        img_bgr = read_bgr(img_path)

        prompt_box = heuristic_bbox_prompt(img_bgr)
        if prompt_box is None:
            return visual_items

        inputs = sam_processor(
            pil_img,
            input_boxes=[[prompt_box]],
            return_tensors="pt",
        ).to(DEVICE)

        with torch.no_grad():
            outputs = sam_model(**inputs)

        masks = sam_processor.image_processor.post_process_masks(
            outputs.pred_masks.cpu(),
            inputs["original_sizes"].cpu(),
            inputs["reshaped_input_sizes"].cpu(),
        )

        mask_tensor = masks[0]

        # Usually shape: [num_boxes, num_masks, H, W]
        if mask_tensor.ndim == 4:
            if hasattr(outputs, "iou_scores") and outputs.iou_scores is not None:
                ious = outputs.iou_scores[0, 0].detach().cpu().numpy()
                best_idx = int(np.argmax(ious))
            else:
                best_idx = 0

            mask = mask_tensor[0, best_idx].numpy()

        elif mask_tensor.ndim == 3:
            mask = mask_tensor[0].numpy()

        else:
            return visual_items

        mask_u8 = (mask > 0.5).astype(np.uint8) * 255

        # Clean mask
        kernel = np.ones((5, 5), np.uint8)
        mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_OPEN, kernel)
        mask_u8 = cv2.morphologyEx(mask_u8, cv2.MORPH_CLOSE, kernel)

        bbox = bbox_from_mask(mask_u8, min_area_ratio=0.001)

        if bbox is None:
            return visual_items

        # Save mask
        mask_name = f"{img_id}_sam_mask.png"
        mask_path = VIS_DIR / mask_name
        cv2.imwrite(str(mask_path), mask_u8)

        # Save overlay
        overlay = make_mask_overlay(img_bgr, mask_u8, bbox=bbox)
        overlay_name = f"{img_id}_sam_overlay.png"
        overlay_path = VIS_DIR / overlay_name
        cv2.imwrite(str(overlay_path), overlay)

        # Save bbox JSON
        bbox_name = f"{img_id}_sam_bbox.json"
        bbox_path = VIS_DIR / bbox_name

        save_json(
            bbox_path,
            {
                "img_id": img_id,
                "bbox_xyxy": bbox,
                "prompt_box_xyxy": prompt_box,
                "source": "SAM fine-tuned on Kvasir-SEG, prompted by heuristic ROI box",
            },
        )

        visual_items.append({
            "type": "segmentation_mask",
            "data": f"visuals/{mask_name}",
            "description": "Polyp or lesion segmentation mask generated by a SAM model fine-tuned on Kvasir-SEG."
        })

        visual_items.append({
            "type": "mask_overlay",
            "data": f"visuals/{overlay_name}",
            "description": "Segmentation mask overlay showing the localized region supporting the visual explanation."
        })

        visual_items.append({
            "type": "bounding_box",
            "data": f"visuals/{bbox_name}",
            "description": "Bounding box derived from the segmentation mask."
        })

    except Exception as e:
        print(f"⚠️ SAM failed for {img_id}: {repr(e)}")

    return visual_items

# ============================================================
# Load S2 results
# ============================================================
with open(s2_json_path, "r", encoding="utf-8") as f:
    s2_results = json.load(f)

if MAX_VISUAL_SAMPLES is not None:
    work_rows = s2_results[:MAX_VISUAL_SAMPLES]
else:
    work_rows = s2_results

print(f"\nTotal S2 rows     : {len(s2_results):,}")
print(f"Rows to process   : {len(work_rows):,}")
print(f"Visual output dir : {VIS_DIR}")

# Show first few image lookup results
print("\nImage lookup sanity check:")
for row in s2_results[:10]:
    img_id = str(row["img_id"])
    p = find_image_path(img_id)
    print(f"  {img_id} -> {p}")

# ============================================================
# Main loop
# ============================================================
updated_entries = []

counts = {
    "sam_visuals": 0,
    "skipped_negative_or_no_trigger": 0,
    "sam_no_mask": 0,
    "missing_image": 0,
}

for row in tqdm(work_rows, desc="Generating visual explanations"):
    img_id = str(row["img_id"])
    img_path = find_image_path(img_id)

    answer = str(row.get("answer", ""))
    question = str(row.get("question", ""))
    explanation = str(row.get("textual_explanation", ""))

    visual_explanation = []

    if img_path is None:
        counts["missing_image"] += 1

    elif not should_generate_visuals(answer, explanation, question):
        counts["skipped_negative_or_no_trigger"] += 1

    else:
        visual_explanation = run_sam_polyp_segmentation(img_path, img_id)

        if visual_explanation:
            counts["sam_visuals"] += 1
        else:
            counts["sam_no_mask"] += 1

    updated_entries.append({
        "val_id": int(row["val_id"]),
        "img_id": row["img_id"],
        "question": row["question"],
        "answer": row["answer"],
        "textual_explanation": row["textual_explanation"],
        "visual_explanation": visual_explanation,
        "confidence_score": float(row["confidence_score"]),
    })

# If debugging with MAX_VISUAL_SAMPLES, append the rest without visuals
if MAX_VISUAL_SAMPLES is not None and MAX_VISUAL_SAMPLES < len(s2_results):
    for row in s2_results[MAX_VISUAL_SAMPLES:]:
        updated_entries.append({
            "val_id": int(row["val_id"]),
            "img_id": row["img_id"],
            "question": row["question"],
            "answer": row["answer"],
            "textual_explanation": row["textual_explanation"],
            "visual_explanation": [],
            "confidence_score": float(row["confidence_score"]),
        })

# Sort by val_id to preserve official order
updated_entries = sorted(updated_entries, key=lambda x: x["val_id"])

# ============================================================
# Rewrite official JSONL
# ============================================================
with open(jsonl_path, "w", encoding="utf-8") as f:
    for obj in updated_entries:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

# ============================================================
# Final report
# ============================================================
print("\n✅ Updated submission_task2.jsonl written")
print("✅ Visuals folder:", VIS_DIR)

print("\nVisual explanation summary:")
for k, v in counts.items():
    print(f"  {k:32s}: {v}")

entries_with_visuals = sum(1 for x in updated_entries if x["visual_explanation"])

print(
    f"\nEntries with visuals: "
    f"{entries_with_visuals}/{len(updated_entries)}"
)

print("\nSample visual_explanation entries:")
shown = 0

for x in updated_entries:
    if x["visual_explanation"]:
        print(json.dumps({
            "val_id": x["val_id"],
            "img_id": x["img_id"],
            "visual_explanation": x["visual_explanation"],
        }, indent=2))
        shown += 1

        if shown >= 3:
            break

if shown == 0:
    print("No visual explanations generated. Check image lookup above and trigger logic.")

Deleted old visual files: 978
IMG_DIR     : /content/drive/MyDrive/CSMORGAN_MEDVQA_2026/data/images
IMG_DIR ok  : True
RESULTS_DIR : /content/drive/MyDrive/CSMORGAN_MEDVQA_2026/results
VIS_DIR     : /content/drive/MyDrive/CSMORGAN_MEDVQA_2026/results/visuals
Device      : cuda

Indexing image files under IMG_DIR...
Indexed image files: 6,500
Sample indexed image names:
  cla820gl7s3zz071u3y0y2yxs -> cla820gl7s3zz071u3y0y2yxs.jpg
  cla820gl6s3yf071ugolc8gg0 -> cla820gl6s3yf071ugolc8gg0.jpg
  cla820gl7s3z3071u90yh7f7r -> cla820gl7s3z3071u90yh7f7r.jpg
  cla820gl6s3yn071u1z4u1dh6 -> cla820gl6s3yn071u1z4u1dh6.jpg
  cla820gl6s3y7071u9k638dv6 -> cla820gl6s3y7071u9k638dv6.jpg
  cla820gl6s3xz071ubrzzdd9a -> cla820gl6s3xz071ubrzzdd9a.jpg
  cla820gl7s3zv071u2ak0ee69 -> cla820gl7s3zv071u2ak0ee69.jpg
  cla820gl6s3yj071u0108e4le -> cla820gl6s3yj071u0108e4le.jpg
  cla820gl7s3z7071ugh7p0ke2 -> cla820gl7s3z7071ugh7p0ke2.jpg
  cla820gl7s3zj071uadm3dpxy -> cla820gl7s3zj071uadm3dpxy.jpg

Loading SAM-Kvasi

Generating visual explanations:   0%|          | 0/1000 [00:00<?, ?it/s]


✅ Updated submission_task2.jsonl written
✅ Visuals folder: /content/drive/MyDrive/CSMORGAN_MEDVQA_2026/results/visuals

Visual explanation summary:
  sam_visuals                     : 401
  skipped_negative_or_no_trigger  : 599
  sam_no_mask                     : 0
  missing_image                   : 0

Entries with visuals: 401/1000

Sample visual_explanation entries:
{
  "val_id": 0,
  "img_id": "cl8k2u1q41ehr0832aze8a3c5",
  "visual_explanation": [
    {
      "type": "segmentation_mask",
      "data": "visuals/cl8k2u1q41ehr0832aze8a3c5_sam_mask.png",
      "description": "Polyp or lesion segmentation mask generated by a SAM model fine-tuned on Kvasir-SEG."
    },
    {
      "type": "mask_overlay",
      "data": "visuals/cl8k2u1q41ehr0832aze8a3c5_sam_overlay.png",
      "description": "Segmentation mask overlay showing the localized region supporting the visual explanation."
    },
    {
      "type": "bounding_box",
      "data": "visuals/cl8k2u1q41ehr0832aze8a3c5_sam_bbox.json",

In [ ]:
# ── CELL 10: Build Task 2 ZIP with Visual Explanations ───────────────────────
import json
import zipfile
from pathlib import Path

# ============================================================
# CONFIG
# ============================================================
TEAM_NAME = "CSMorgan_2"
ZIP_PATH  = RESULTS_DIR / f"{TEAM_NAME}_task2.zip"

jsonl_path = RESULTS_DIR / "submission_task2.jsonl"
visuals_dir = RESULTS_DIR / "visuals"

assert jsonl_path.exists(), "Run Cell 09 first to generate submission_task2.jsonl"

# ============================================================
# submission_task2.py: required team metadata
# ============================================================
SUBMISSION_PY = '''
SUBMISSION_INFO = {
    "Participant_Names": "Peter Ojonugwa Ejiga",
    "Affiliations": "Morgan State University, Computer Vision & AI Lab",
    "Contact_emails": ["ojeji1@morgan.edu"],
    "Team_Name": "CSMorgan-MEDVQA",
    "Country": "United States",
    "Notes_to_organizers": (
        "Task 2 submission for ImageCLEFmed MEDVQA-GI 2026. "
        "The system uses Qwen2.5-VL-7B-Instruct QLoRA for Task 1 answer generation "
        "and MedGemma-4B QLoRA for Task 2 clinician-oriented textual explanations. "
        "Visual explanations are generated where reliable grounding is available using "
        "a SAM model fine-tuned on Kvasir-SEG for polyp segmentation. Bounding boxes are "
        "derived from segmentation masks. Entries without reliable localization keep "
        "visual_explanation as an empty list. Confidence scores are capped for conservative "
        "safety calibration."
    ),
}
'''.strip()

# ============================================================
# Load and validate JSONL
# ============================================================
required = [
    "val_id",
    "img_id",
    "question",
    "answer",
    "textual_explanation",
    "visual_explanation",
    "confidence_score",
]

entries = []
errors = []

with open(jsonl_path, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()

        if not line:
            errors.append(f"Line {line_no}: empty line")
            continue

        try:
            obj = json.loads(line)
        except Exception as e:
            errors.append(f"Line {line_no}: invalid JSON: {repr(e)}")
            continue

        missing = [k for k in required if k not in obj]
        if missing:
            errors.append(f"Line {line_no}: missing fields {missing}")
            continue

        # val_id
        if not isinstance(obj["val_id"], int):
            errors.append(f"Line {line_no}: val_id must be int")

        # string fields
        for k in ["img_id", "question", "answer", "textual_explanation"]:
            if not isinstance(obj[k], str) or not obj[k].strip():
                errors.append(f"Line {line_no}: {k} must be a non-empty string")

        # visual_explanation
        if not isinstance(obj["visual_explanation"], list):
            errors.append(f"Line {line_no}: visual_explanation must be a list")
        else:
            for j, item in enumerate(obj["visual_explanation"]):
                if not isinstance(item, dict):
                    errors.append(f"Line {line_no}: visual_explanation[{j}] must be a dict")
                    continue

                if "type" not in item:
                    errors.append(f"Line {line_no}: visual_explanation[{j}] missing type")

                if "data" not in item:
                    errors.append(f"Line {line_no}: visual_explanation[{j}] missing data")

                if "description" not in item:
                    errors.append(f"Line {line_no}: visual_explanation[{j}] missing description")

                # If data is a path, it should be relative to ZIP root.
                data = item.get("data", "")
                if isinstance(data, str) and data.startswith("/"):
                    errors.append(
                        f"Line {line_no}: visual_explanation[{j}] data must be relative, not absolute"
                    )

        # confidence_score
        try:
            conf = float(obj["confidence_score"])
            if not (0.0 <= conf <= 1.0):
                errors.append(f"Line {line_no}: confidence_score outside [0, 1]")
            obj["confidence_score"] = conf
        except Exception:
            errors.append(f"Line {line_no}: confidence_score must be numeric")

        entries.append(obj)

# ============================================================
# Validate val_id uniqueness and completeness
# ============================================================
if entries:
    val_ids = [e["val_id"] for e in entries]

    seen = set()
    duplicate_val_ids = []

    for v in val_ids:
        if v in seen:
            duplicate_val_ids.append(v)
        seen.add(v)

    duplicate_val_ids = sorted(set(duplicate_val_ids))

    if duplicate_val_ids:
        errors.append(f"Duplicate val_id values found: {duplicate_val_ids[:20]}")

    expected = set(range(len(entries)))
    found = set(val_ids)

    missing_val_ids = sorted(expected - found)
    extra_val_ids = sorted(found - expected)

    if missing_val_ids:
        errors.append(f"Missing val_id values: {missing_val_ids[:20]}")

    if extra_val_ids:
        errors.append(f"Unexpected val_id values outside 0..N-1: {extra_val_ids[:20]}")

# ============================================================
# Check visual_explanation referenced files
# ============================================================
visual_paths = []

for obj in entries:
    for item in obj.get("visual_explanation", []):
        if isinstance(item, dict):
            data = item.get("data", "")

            if isinstance(data, str) and data.startswith("visuals/"):
                visual_paths.append(data)

missing_visuals = []

for rel_path in visual_paths:
    local_path = RESULTS_DIR / rel_path
    if not local_path.exists():
        missing_visuals.append(rel_path)

if missing_visuals:
    errors.append(
        "visual_explanation references missing files: "
        + ", ".join(missing_visuals[:20])
    )

# ============================================================
# Stop if validation failed
# ============================================================
if errors:
    print("❌ Validation failed:")
    for e in errors[:80]:
        print(" -", e)

    if len(errors) > 80:
        print(f" ... plus {len(errors) - 80} more errors")

    raise ValueError(f"Task 2 validation failed with {len(errors)} error(s).")

# ============================================================
# Summary before ZIP
# ============================================================
num_entries = len(entries)
num_with_visuals = sum(1 for e in entries if e.get("visual_explanation"))
num_visual_refs = len(visual_paths)

print(f"✅ JSONL valid: {num_entries:,} entries")
print("✅ Required fields present")
print("✅ val_id values are unique and complete")
print("✅ confidence_score values are in [0, 1]")
print(f"✅ Entries with visual explanations: {num_with_visuals:,}/{num_entries:,}")
print(f"✅ Referenced visual files: {num_visual_refs:,}")

if num_visual_refs > 0:
    print("✅ All referenced visual files exist")
else:
    print("ℹ️ No visual files referenced. This is still valid because visual_explanation is optional.")

# ============================================================
# README.md
# ============================================================
README = f"""# {TEAM_NAME} — ImageCLEFmed MEDVQA-GI 2026 Task 2

## ZIP Layout

This archive follows the required Task 2 layout:

```text
{TEAM_NAME}_task2.zip
├── submission_task2.jsonl
├── submission_task2.py
├── visuals/
└── README.md
"""

✅ JSONL valid: 1,000 entries
✅ Required fields present
✅ val_id values are unique and complete
✅ confidence_score values are in [0, 1]
✅ Entries with visual explanations: 401/1,000
✅ Referenced visual files: 1,203
✅ All referenced visual files exist
